# Hugging Face Embeddings — Direct Client + LangChain

This notebook replicates the experiments from the **Ollama Embeddings** notebook using **Hugging Face Inference** and `langchain_huggingface`.

We keep the same overall flow:

```text
Query → embedding → inspect dimensions
PDF → pages → chunks → document embeddings
                     ↓
             LangChain embeddings
```

**Cloud setup:** the direct client uses Hugging Face's `InferenceClient`, while the LangChain section uses `HuggingFaceEndpointEmbeddings`, which uses the Hugging Face client underneath.


In [1]:
%pip install -U huggingface_hub langchain-huggingface langchain-text-splitters langchain-community pypdf python-dotenv


   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ------------------------------------- -- 786.4/846.4 kB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 846.4/846.4 kB 5.5 MB/s  0:00:00

  Attempting uninstall: pypdf

    Found existing installation: pypdf 6.18.1

    Uninstalling pypdf-6.18.1:

      Successfully uninstalled pypdf-6.18.1

   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]


## 1. Imports and configuration

For this notebook we use a hosted Hugging Face embedding model so that the embedding computation happens through Hugging Face Inference rather than downloading the model to the local machine.


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEndpointEmbeddings

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "Hugging Face token not found. Add HF_TOKEN=... to your .env file."
    )

MODEL = "BAAI/bge-small-en-v1.5"
PROVIDER = "hf-inference"

print(f"Model: {MODEL}")
print(f"Provider: {PROVIDER}")


C:\Users\Uttam\AppData\Local\Temp\ipykernel_8424\2451426571.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Model: BAAI/bge-small-en-v1.5
Provider: hf-inference


### `.env`

Create/update your `.env` file:

```text
HF_TOKEN=your_hugging_face_token
```

Do **not** commit `.env` to Git.


## HUGGING FACE — Direct Python Client

This section replaces the direct `ollama.embed(...)` experiments.

Hugging Face's official Python client is `huggingface_hub.InferenceClient`. Its `feature_extraction()` method converts text into embedding vectors and accepts either a single string or a list of strings.


In [3]:
client = InferenceClient(
    provider=PROVIDER,
    api_key=HF_TOKEN
)


In [4]:
query = "What is Openclaw/Moltbot and what are the major security concerns regarding this tool"

query_embeddings = client.feature_extraction(
    query,
    model=MODEL
)

query_embeddings


array([ 4.59491927e-03, -5.80920801e-02, -3.98145095e-02, -1.71330571e-02,
        3.37630324e-02, -7.04948083e-02,  5.98530844e-02,  1.15504125e-02,
       -5.59421740e-02, -2.12798058e-03,  5.99879250e-02, -3.57355699e-02,
        2.91170622e-03,  2.74204742e-02,  2.05866080e-02,  2.44660843e-02,
        1.86167341e-02, -3.00980755e-03,  3.99259031e-02,  1.62093006e-02,
        5.53651974e-02,  1.66022722e-02,  2.41188779e-02, -4.85500619e-02,
       -6.32168502e-02,  5.74002452e-02, -3.81322652e-02, -2.46305596e-02,
       -7.46413320e-02, -1.73465326e-01, -1.03713978e-06, -5.57660758e-02,
        1.30259693e-02, -2.56257840e-02,  1.16892047e-02,  2.69533787e-02,
        3.14364880e-02,  2.14026738e-02, -2.46304665e-02, -1.68271698e-02,
        5.32033993e-03,  4.92882021e-02, -4.31912728e-02, -2.08307616e-03,
        1.83113422e-02, -7.49871060e-02,  4.98679429e-02, -9.97905061e-03,
        8.56073573e-03, -7.33377365e-03, -4.81276624e-02, -2.60214806e-02,
        1.94289908e-02,  

In [5]:
# dimensions

print(query_embeddings.shape)
print("Embedding dimensions:", query_embeddings.shape[-1])


(384,)
Embedding dimensions: 384


### Dimension experiment

The original Ollama notebook explicitly requested:

```python
dimensions=512
```

Hugging Face's `feature_extraction()` API does **not** expose that as a general dimension-reduction switch for ordinary feature-extraction models. The current API documents `dimensions` only for OpenAI-compatible embedding endpoints.

Therefore, instead of forcing 512 dimensions, this notebook records the **native output dimension of the selected Hugging Face model**.

For `BAAI/bge-small-en-v1.5`, the native embedding size is expected to be 384.


In [6]:
# Native dimension of the selected model

print("Native embedding dimensions:", query_embeddings.shape[-1])


Native embedding dimensions: 384


## 2. Load the same PDF

This is intentionally kept the same as the Ollama notebook so the embedding experiments are comparable.

Expected file:

```text
../Openclaw_Research_Report.pdf
```


In [7]:
# create the loader

loader = PyPDFLoader(file_path="../Openclaw_Research_Report.pdf")

docs = loader.load()

len(docs)


34

## 3. Split documents

Same splitter configuration as the Ollama notebook:

- `chunk_size=300`
- `chunk_overlap=50`

This isolates the embedding-provider change from the chunking configuration.


In [8]:
# split documents

chunker = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = chunker.split_documents(docs)

len(chunks)


272

In [9]:
# embed documents

text_documents = [doc.page_content for doc in chunks]

print(len(text_documents))


272


## 4. Direct Hugging Face document embeddings

The Ollama notebook passed the complete list of chunk texts to `ollama.embed()`.

Here we do the equivalent using `InferenceClient.feature_extraction()`.


In [10]:
# embed documents

document_embeddings = client.feature_extraction(
    text_documents,
    model=MODEL
)

document_embeddings


array([[-0.04975554, -0.05581394, -0.04917482, ...,  0.02149331,
         0.00921732, -0.00099139],
       [-0.02629246, -0.00525718, -0.01360095, ...,  0.05997259,
         0.02286262, -0.02067402],
       [-0.02025921,  0.00105199,  0.03318819, ..., -0.0238363 ,
         0.04467135,  0.04586597],
       ...,
       [-0.03499734, -0.0547357 , -0.02714531, ...,  0.03947995,
        -0.01328703,  0.00642882],
       [-0.04854969, -0.03223035, -0.00666501, ...,  0.04352531,
         0.00901671, -0.01849797],
       [-0.04688414, -0.06458104, -0.02152348, ...,  0.04329114,
        -0.04348437,  0.01284446]], shape=(272, 384), dtype=float32)

In [11]:
# number of embedding vectors

print("Number of embedding vectors:", len(document_embeddings))


Number of embedding vectors: 272


In [12]:
# output dimensions

print("Embedding dimensions:", document_embeddings.shape[-1])


Embedding dimensions: 384


## LANGCHAIN HUGGING FACE

Now we repeat the final section of the Ollama notebook using LangChain's official Hugging Face integration.

`HuggingFaceEndpointEmbeddings` is the hosted/cloud integration. It uses the Hugging Face client underneath rather than running the embedding model locally.


In [13]:
# create the embedder

langchain_embedder = HuggingFaceEndpointEmbeddings(
    model=MODEL,
    task="feature-extraction",
    provider=PROVIDER,
    huggingfacehub_api_token=HF_TOKEN
)


In [14]:
# embed documents

langchain_document_embeddings = langchain_embedder.embed_documents(
    texts=text_documents
)


In [15]:
# number of embedding vectors

len(langchain_document_embeddings)


272

In [16]:
# output dimensions

len(langchain_document_embeddings[0])


384

## 5. Compare Direct Client vs LangChain

Both paths should produce the same basic shape:

```text
number of chunks × embedding dimension
```

For this notebook:

```text
273 chunks × 384 dimensions
```

if your PDF produces the same 273 chunks as the original notebook and the selected model is `BAAI/bge-small-en-v1.5`.

The exact chunk count depends on the PDF extraction/splitting result.


In [17]:
print("Direct client:")
print("  vectors:", len(document_embeddings))
print("  dimensions:", document_embeddings.shape[-1])

print("\nLangChain:")
print("  vectors:", len(langchain_document_embeddings))
print("  dimensions:", len(langchain_document_embeddings[0]))


Direct client:
  vectors: 272
  dimensions: 384

LangChain:
  vectors: 272
  dimensions: 384


## 6. Final architecture

```text
                         Hugging Face

                  ┌──────────────────────┐
                  │  InferenceClient     │
                  │  feature_extraction  │
                  └──────────┬───────────┘
                             │
                             ▼
                         Embeddings
                             │
PDF → PyPDFLoader → RecursiveCharacterTextSplitter
                             │
                             ▼
                      Chunk embeddings
                             │
                             ▼
                           Qdrant
                             │
                             ▼
                        RAG retrieval


LangChain path:

HuggingFaceEndpointEmbeddings
            │
            └── uses Hugging Face inference client
```

### Ollama → Hugging Face mapping

| Ollama notebook | Hugging Face notebook |
|---|---|
| `ollama.embed()` | `InferenceClient.feature_extraction()` |
| `OllamaEmbeddings` | `HuggingFaceEndpointEmbeddings` |
| `embeddinggemma` | `BAAI/bge-small-en-v1.5` |
| Local Ollama server | Hugging Face Inference |
| `dimensions=512` | Native model dimension |
| `embed_documents()` | `embed_documents()` |

The important conceptual point is that **LangChain's embedding interface stays the same**:

```python
embed_query(...)
embed_documents(...)
```

Only the provider-specific implementation changes.


## Important note about dimensions

Do not treat embedding dimension as an arbitrary setting.

For a normal embedding model, the vector dimension is part of the model's output space. If you later put these vectors into Qdrant, the Qdrant collection must be configured for the same dimension.

For example:

```text
BAAI/bge-small-en-v1.5
        ↓
      384-d
        ↓
Qdrant collection size = 384
```

If you change the embedding model to one that outputs 768 dimensions, you need a compatible Qdrant collection/configuration.
